# A100: time one epoch of SGD, SAM and IGR before full training
**This notebook runs exactly one measured training epoch per method. It never launches the 100-epoch / three-seed experiment.** Each method has sacrificial warm-up updates and a short timing probe first; their model/BN/optimizer state is discarded before the full measured epoch.

Select **A100 with at least 38,000 MiB GPU memory**. Works in Colab; on the course's Linux Jupyter A100, set `WORK_ROOT` to your persistent mounted directory (for example `/workspace/sam_igr_a100_timing`) and leave `USE_DRIVE=False`.

| Setting | Timing run |
|---|---|
| Data | CIFAR-100: balanced 45,000 training / 5,000 validation |
| Model | Same CIFAR ResNet-18 as notebooks 06/07, 100 outputs, FP32 |
| Batch | 128, one GPU, 351 full updates; 44,928 images used / 72 dropped |
| Measured training | One epoch EACH of SGD, SAM and IGR; seed 42 |
| Warm-up | 10 updates + 20 timed probe updates, then reset |
| Optimizer | Ordinary SGD momentum 0.9, Nesterov off; LR 0.1; L2 0.0005; clipping 5 |
| Schedule | Epoch 1 of a 100-epoch cosine schedule |
| SAM / IGR | rho 0.05 / lambda 0.01 (timing value, not a new lambda search) |
| Evaluation | All 5,000 validation images; official test images never loaded |
| Checkpoint | Optional optimizer + BN write and exact tensor roundtrip, separately timed |

The batch changes from 32 in the T4 pilots to 128 here, affecting both BatchNorm and SAM's perturbation batch. Only extrapolate to a full experiment with this exact batch/data/precision/protocol. If IGR runs out of memory, the attempt stops and preserves its log; it does not silently change settings. A100 execution of this new notebook remains unverified.


In [ ]:
RUN_TIMING = True  # Executes ONLY this bounded one-epoch timing experiment.
USE_DRIVE = False  # Optional Colab-only output persistence; checkpoints can be large.
WORK_ROOT = '/content/sam_igr_t4_pilot'  # Reuses the existing legacy environment/data cache if present.
MAX_SESSION_MINUTES = 60  # Shared subprocess deadline, including setup/data preparation and all methods.
MEASURE_CHECKPOINT = True

# Planning scenario ONLY. These values never control the worker's epoch loop.
PLANNED_EPOCHS = 100
PLANNED_SEEDS = 3
PLANNED_IGR_SEARCH_TRIALS = 3
PLANNED_IGR_SEARCH_EPOCHS = 100
PLANNED_IGR_SEARCH_SEEDS = 1
EXTRA_EVALUATION_RESERVE_MINUTES = 60  # Unmeasured total reserve for later PGD/sharpness/test analysis.
ALLOWANCE_FRACTION = 0.20
PLANNING_BUDGET_HOURS = 35  # Planning comparison only; not the allocation/account balance.


## 1. Isolated environment
Reuses the pinned Python 3.8 / JAX 0.2.6 / CUDA jaxlib 0.1.57 / Flax 0.2.2 bootstrap from notebook 07, with the corrected micromamba `-c` handling. Colab's own Python is unchanged. No restart is requested. Source for the imported ResNet block is pinned to the existing group commit.

All subprocesses share a one-hour deadline from this setup cell. Installation and the CIFAR download can consume most of that hour; if it expires, preserve the failed folder and rerun setup to create a fresh attempt using the downloaded cache/environment. This does not disconnect the GPU or cap billing. Concurrent notebooks sharing the same environment should not be running during setup/measurement.


In [ ]:
import os, sys, json, time, hashlib, subprocess, shutil, tarfile, io, urllib.request
import threading, signal, uuid, csv, traceback, platform, fcntl
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path(WORK_ROOT)
DRIVE = Path('/content/drive/MyDrive/sam_replication')
SOURCE_REVISION = '7f34662deec03341b78cccd4ce18ec2f7900429b'
REPO = ROOT / 'work' / 'sam'; PREFIX = ROOT / 'envs' / 'sam'; PYTHON = PREFIX / 'bin' / 'python'
SESSION = None

def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False)); os.replace(str(tmp), str(path))

def command(args, log, environment=None, deadline=None, check=True):
    """Stream file logs; bound subprocesses even if the notebook kernel disappears."""
    args = list(map(str, args))
    # micromamba uses -c for channels; only Python's own -c contains code.
    python_inline = len(args) > 2 and Path(args[0]).name.startswith('python') and args[1] == '-c'
    if python_inline:
        compile(args[2], '<subprocess Python>', 'exec')
    log = Path(log); log.parent.mkdir(parents=True, exist_ok=True)
    remaining = int((deadline or (time.monotonic()+1800))-time.monotonic())
    if remaining <= 0: raise TimeoutError('Session deadline reached; no new process started')
    bounded = ['timeout', '--signal=INT', '--kill-after=15s', str(remaining)+'s']+args
    label = list(args[:5])
    if python_inline:
        label[2] = '<embedded Python>'
    print('Running:', ' '.join(label), flush=True)
    with log.open('a') as output:
        output.write('\nCOMMAND '+repr(args)+'\n'); output.flush()
        offset = log.stat().st_size
        process = subprocess.Popen(bounded, stdout=output, stderr=subprocess.STDOUT,
                                   env=environment, start_new_session=True)
        try:
            with log.open() as reader:
                reader.seek(offset); heartbeat = time.monotonic()
                while process.poll() is None:
                    chunk = reader.read()
                    if chunk: print(chunk, end='', flush=True)
                    if time.monotonic()-heartbeat > 30:
                        print('[still running; log:', log.name+']', flush=True); heartbeat = time.monotonic()
                    time.sleep(1)
                print(reader.read(), end='', flush=True)
            if check and process.returncode: raise RuntimeError('Exit %s; see %s' % (process.returncode, log))
            return process.returncode
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGINT)
                try: process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL); process.wait()

def environment():
    env = os.environ.copy()
    env.update(PATH=str(PREFIX/'bin')+':'+env.get('PATH', ''),
        LD_LIBRARY_PATH=str(PREFIX/'lib')+':'+env.get('LD_LIBRARY_PATH', ''),
        PYTHONPATH=str(REPO.parent), CUDA_HOME=str(PREFIX), CUDA_PATH=str(PREFIX),
        CUDA_VISIBLE_DEVICES='0', XLA_PYTHON_CLIENT_PREALLOCATE='false', PYTHONHASHSEED='0',
        TFDS_DATA_DIR=str(ROOT/'tensorflow_datasets'), TF_DETERMINISTIC_OPS='0',
        TF_CUDNN_DETERMINISTIC='0',
        XLA_FLAGS='--xla_gpu_cuda_data_dir='+str(ROOT/'cuda')+
            ' --xla_gpu_deterministic_reductions='+'false')
    return env

VERSION_CHECK = '''
import sys
from importlib.metadata import version
assert sys.version_info[:2] == (3,8)
expected={'jax':'0.2.6','jaxlib':'0.1.57+cuda110','flax':'0.2.2','numpy':'1.18.5','tensorflow':'2.3.1'}
for package,want in expected.items():
    got=version(package); assert got==want,(package,got,want)
import numpy, tensorflow, flax, jax
print('Verified pinned packages:',expected,flush=True)
'''
GPU_CHECK = VERSION_CHECK + '''
import tensorflow as tf
tf.config.experimental.set_visible_devices([], 'GPU')
assert len(jax.devices())==1 and jax.devices()[0].platform=='gpu'
x=jax.jit(lambda a:a@a)(jax.numpy.ones((2,2))); x.block_until_ready()
print('GPU initialization PASS:',jax.devices(),flush=True)
'''

def pinned_cuda_ready():
    packages = {}
    for path in (PREFIX/'conda-meta').glob('*.json'):
        record = json.loads(path.read_text())
        packages[record['name']] = record['version']
    return (packages.get('cudatoolkit', '').startswith('11.0.') and
            packages.get('cudnn', '').split('.')[:3] == ['8', '0', '5'] and packages.get('cuda-nvcc') == '11.3.122')

def setup(deadline):
    if not REPO.exists():
        REPO.parent.mkdir(parents=True, exist_ok=True)
        command(['git', 'clone', '--branch', 'repro', 'https://github.com/Siddharth-magesh/sam-replication.git', REPO], SESSION/'setup.log', deadline=deadline)
        command(['git', '-C', REPO, 'checkout', '--detach', SOURCE_REVISION], SESSION/'setup.log', deadline=deadline)
    actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True, timeout=30).strip()
    assert actual == SOURCE_REVISION, 'Existing runtime checkout differs; use a fresh ROOT.'
    assert not subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain', '--untracked-files=no'], text=True, timeout=30).strip(), 'Tracked source modifications'
    ready = False
    if PYTHON.exists() and (PREFIX/'bin'/'ptxas').exists() and pinned_cuda_ready():
        ready = command([PYTHON, '-c', VERSION_CHECK], SESSION/'environment_check.log', environment(), deadline, check=False) == 0
    if not ready:
        manager = ROOT/'bin'/'micromamba'
        if not manager.exists():
            with urllib.request.urlopen('https://micro.mamba.pm/api/micromamba/linux-64/latest', timeout=120) as response:
                data = response.read()
            with tarfile.open(fileobj=io.BytesIO(data), mode='r:bz2') as archive:
                member = archive.getmember('bin/micromamba'); assert member.isfile()
                manager.parent.mkdir(parents=True, exist_ok=True)
                manager.write_bytes(archive.extractfile(member).read())
            manager.chmod(0o755)
        os.environ['MAMBA_ROOT_PREFIX'] = str(ROOT/'mamba')
        action = 'install' if PYTHON.exists() else 'create'
        command([manager, action, '-p', PREFIX, '-c', 'conda-forge', '--override-channels',
                 'python=3.8', 'pip=24.0', 'cudatoolkit=11.0', 'cudnn=8.0.5', '-y'], SESSION/'setup.log', deadline=deadline)
        command([manager, 'install', '-p', PREFIX, '-c', 'nvidia/label/cuda-11.3.1', '--override-channels',
                 'cuda-nvcc=11.3.122', '-y'], SESSION/'setup.log', deadline=deadline)
        lines = (REPO/'requirements.txt').read_text().splitlines()
        requirements = ROOT/'requirements_runtime.txt'
        requirements.write_text('\n'.join(s.replace('absl-py==0.11.0', 'absl-py==0.10.0')
            for s in lines if not s.startswith('jaxlib'))+'\n')
        command([PYTHON, '-m', 'pip', 'install', 'jaxlib==0.1.57+cuda110', '-f',
                 'https://storage.googleapis.com/jax-releases/jax_cuda_releases.html'], SESSION/'setup.log', deadline=deadline)
        command([PYTHON, '-m', 'pip', 'install', '-r', requirements], SESSION/'setup.log', deadline=deadline)
    assert pinned_cuda_ready(), 'CUDA/cuDNN pins differ; inspect setup log instead of upgrading JAX'
    atomic_json(SESSION/'cuda_packages.json', [json.loads(p.read_text()) for p in (PREFIX/'conda-meta').glob('*.json')
        if p.name.startswith(('cudatoolkit-', 'cudnn-', 'cuda-nvcc-'))])
    candidates = sorted(PREFIX.rglob('libdevice*.bc')); assert candidates, 'CUDA libdevice missing'
    libdir = ROOT/'cuda'/'nvvm'/'libdevice'; libdir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(candidates[0]), str(libdir/'libdevice.10.bc'))
    command([PREFIX/'bin'/'ptxas', '--version'], SESSION/'environment_check.log', environment(), deadline)
    command([PYTHON, '-c', GPU_CHECK], SESSION/'environment_check.log', environment(), deadline)
    command([PYTHON, '-m', 'pip', 'freeze'], SESSION/'package_versions.txt', environment(), deadline)



assert sys.platform=='linux' and platform.machine()=='x86_64', 'Use Linux x86_64 with an NVIDIA A100'
assert 0<MAX_SESSION_MINUTES<=60
SESSION_STARTED=time.monotonic();DEADLINE=SESSION_STARTED+MAX_SESSION_MINUTES*60
GPU=subprocess.check_output(['nvidia-smi','-i','0','--query-gpu=name,memory.total',
    '--format=csv,noheader,nounits'],text=True,timeout=15).strip()
name,memory=GPU.rsplit(',',1)
assert 'A100' in name and float(memory)>=38000, 'Use A100 >= 38,000 MiB; no silent T4/slice/batch fallback.'
print('Timing GPU:',GPU)
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT=Path('/content/drive/MyDrive/sam_replication/igr_a100_one_epoch')
else:
    OUTPUT_ROOT=ROOT/'a100_timing_runs'
SESSION=OUTPUT_ROOT/(datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:6])
SESSION.mkdir(parents=True,exist_ok=False)
print('TIMING RESULTS:',SESSION,flush=True)
t=time.monotonic()
try:setup(min(DEADLINE,time.monotonic()+1800))
finally:atomic_json(SESSION/'setup_timing.json',dict(seconds=time.monotonic()-t))
ENV=environment()
print('Pinned environment ready. Full experiment will NOT be launched.')


## 2. Exact timing worker and dataset preparation
SGD/SAM/IGR objectives and the ResNet architecture are copied unchanged from notebook 07. IGR uses exact mixed derivatives and frozen BN in its input-gradient penalty branch. The clean CE branch updates BN; SAM retains clean first-pass BN state.

Dataset download/preparation is a separate host phase, before training measurements. Every method runs in a fresh subprocess so previous method allocations/executables are released. All methods must have the same initial model/BN hash and epoch-order hash before a combined projection is accepted.


In [ ]:
WORKER_SOURCE = r'''



# Intermediate derived from notebook 06 and notebook 02's JAX/Flax CIFAR ResNet pilot. See notebook protocol.
import os, sys, json, time, math, hashlib, platform, subprocess, traceback, gc
BEGIN = time.monotonic()
from pathlib import Path
import numpy as np
import tensorflow as tf
tf.config.experimental.set_visible_devices([], 'GPU')
import tensorflow_datasets as tfds
import jax
import jax.numpy as jnp
import flax
from flax import nn, optim, serialization
C = json.loads(Path(sys.argv[1]).read_text())
OUT = Path(C['output']); PHASE = sys.argv[2]
sys.path.insert(0, C['repo'])
from sam_jax.imagenet_models.resnet import ResNetBlock
B = C['batch_size']; METHODS = ('SGD','SAM','IGR')

def ready(tree):
    for v in jax.tree_leaves(tree):
        if hasattr(v, 'block_until_ready'): v.block_until_ready()
    return tree

def save(name, value):
    path=OUT/name; tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False)); tmp.replace(path)

def gpu_memory():
    try:
        raw=subprocess.check_output(['nvidia-smi','-i','0','--query-gpu=name,memory.used,memory.total',
            '--format=csv,noheader,nounits'],text=True,timeout=10).strip()
        name,used,total=raw.rsplit(',',2)
        return dict(name=name.strip(),used_mib=float(used),total_mib=float(total),
                    measurement='nvidia-smi snapshot, not true peak')
    except Exception as exc:
        return dict(error=str(exc))

def check_time():
    if time.monotonic()-BEGIN > C['worker_seconds']:
        raise TimeoutError('Worker time cap reached; partial runs cannot enter the final comparison.')

def finite(tree):
    assert all(np.isfinite(np.asarray(x)).all() for x in jax.tree_leaves(tree)), 'Non-finite value'

def norm2(tree):
    return sum(jnp.sum(x*x) for x in jax.tree_leaves(tree))

def l2(tree): return jnp.sqrt(norm2(tree))

def add(a,b): return jax.tree_multimap(lambda x,y:x+y,a,b)
def sub(a,b): return jax.tree_multimap(lambda x,y:x-y,a,b)
def mul(a,s): return jax.tree_map(lambda x:x*s,a)

assert jax.__version__=='0.2.6' and flax.__version__=='0.2.2'
assert len(jax.devices())==1 and jax.devices()[0].platform=='gpu', 'One NVIDIA GPU required'
# Seeds for splits, augmentation and initialization are explicit and separate.
builder=tfds.builder('cifar100:3.0.2', data_dir=C['data_dir'])
assert builder.info.splits, 'Prepare CIFAR-100 in the preceding data cell; worker never downloads data.'
assert builder.info.splits['train'].num_examples==50000
train=tfds.as_numpy(builder.as_dataset(split='train',batch_size=-1))
images,labels=train['image'],train['label'].astype(np.int32)
rs=np.random.RandomState(C['split_seed']); ti=[]; vi=[]
for cls in range(100):
    ids=rs.permutation(np.flatnonzero(labels==cls))
    vi.extend(ids[:50]); ti.extend(ids[50:50+C['train_per_class']])
ti=np.array(ti,dtype=np.int32); vi=np.array(vi,dtype=np.int32)
assert len(set(ti.tolist()) & set(vi.tolist()))==0
assert len(ti)==C['train_per_class']*100 and len(vi)==5000
np.savez(OUT/'split_indices.npz',train=ti,validation=vi)
STEPS=len(ti)//B
assert STEPS>0

def batches(epoch, seed):
    rng=np.random.RandomState(seed+1000+epoch); order=rng.permutation(ti)
    for off in range(0,STEPS*B,B):
        ids=order[off:off+B]
        padded=np.pad(images[ids],((0,0),(4,4),(4,4),(0,0)))
        xy=rng.randint(0,9,(B,2)); flip=rng.rand(B)<0.5
        x=np.stack([padded[i,u:u+32,v:v+32] for i,(u,v) in enumerate(xy)])
        x[flip]=x[flip,:,::-1,:]
        yield x.astype(np.float32)/255.,labels[ids]

def eval_batches(xx, yy, ids):
    for off in range(0,len(ids),B):
        chunk=ids[off:off+B]; n=len(chunk)
        x=np.zeros((B,32,32,3),np.float32); y=np.zeros(B,np.int32)
        x[:n]=xx[chunk].astype(np.float32)/255.; y[:n]=yy[chunk]
        mask=(np.arange(B)<n).astype(np.float32)
        yield x,y,mask

class CifarResNet18(nn.Module):
    def apply(self,x,train=True):
        # Raw [0,1] pixel coordinates for IGR and attacks, normalization INSIDE model.
        x=(x-jnp.array([0.5071,0.4867,0.4408]))/jnp.array([0.2675,0.2565,0.2761])
        conv=nn.Conv.partial(bias=False,dtype=jnp.float32)
        bn=nn.BatchNorm.partial(use_running_average=not train,momentum=0.9,epsilon=1e-5,dtype=jnp.float32)
        x=nn.relu(bn(conv(x,64,(3,3),(1,1),padding='SAME',name='stem'),name='stem_bn'))
        for stage in range(4):
            for block in range(2):
                stride=(2,2) if stage>0 and block==0 else (1,1)
                x=ResNetBlock(x,64*2**stage,conv=conv,norm=bn,act=nn.relu,strides=stride,
                              name='stage%d_block%d'%(stage,block))
        return nn.Dense(jnp.mean(x,axis=(1,2)),100)

def fresh(seed):
    with nn.stateful() as state:
        _,params=CifarResNet18.init_by_shape(jax.random.PRNGKey(seed),[((B,32,32,3),jnp.float32)])
    model=nn.Model(CifarResNet18,params)
    return optim.Momentum(learning_rate=C['lr'],beta=C['momentum'],nesterov=False).create(model),state

def losses(logits,y):
    return -jnp.take_along_axis(jax.nn.log_softmax(logits),y[:,None],axis=1)[:,0]

def predict(model,state,x):
    with nn.stateful(state,mutable=False): return model(x,train=False)

def train_loss(model,state,x,y):
    with nn.stateful(state) as ns: z=model(x,train=True)
    ce=jnp.mean(losses(z,y))
    return ce+0.5*C['weight_decay']*norm2(model.params),(ns,ce)

# With frozen BN there is no cross-example dependence. Gradient of the SUM
# returns each image's CE gradient, without the 1/B scaling of gradient(mean CE).
def input_grads(model,state,x,y):
    return jax.grad(lambda pixels:jnp.sum(losses(predict(model,state,pixels),y)))(x)

def penalty(model,state,x,y):
    dx=input_grads(model,state,x,y)
    return jnp.mean(jnp.sum(dx*dx,axis=(1,2,3)))

def igr_loss(model,state,x,y,lam):
    base,(ns,ce)=train_loss(model,state,x,y)
    # Differentiate THROUGH dx to parameters. Only BN running state is constant.
    p=penalty(model,state,x,y)
    return base+lam*p,(ns,ce,p)

def make_step(method):
    @jax.jit
    def step(opt,state,x,y,lr,lam):
        if method=='IGR':
            (value,(ns,ce,p)),g=jax.value_and_grad(igr_loss,has_aux=True)(opt.target,state,x,y,lam)
        else:
            (value,(ns,ce)),g=jax.value_and_grad(train_loss,has_aux=True)(opt.target,state,x,y)
            p=jnp.float32(0.)
            if method=='SAM':
                delta=mul(g,C['rho']/jnp.maximum(l2(g),1e-12))
                perturbed=add(opt.target,delta)
                (_, _),g=jax.value_and_grad(train_loss,has_aux=True)(perturbed,state,x,y)
                # Update ORIGINAL weights; keep clean first-pass BN statistics only.
        gn=l2(g); g=mul(g,jnp.minimum(1.,C['clip']/jnp.maximum(gn,1e-12)))
        return opt.apply_gradient(g,learning_rate=lr),ns,dict(objective=value,clean_ce=ce,igr_penalty=p,gradient_norm=gn)
    return step
STEPS_FN={m:make_step(m) for m in METHODS}

@jax.jit
def clean_batch(model,state,x,y,mask):
    z=predict(model,state,x)
    return jnp.sum((jnp.argmax(z,axis=1)==y)*mask),jnp.sum(losses(z,y)*mask)

def evaluate(model,state,xx,yy,ids):
    acc=ce=0.
    for x,y,mask in eval_batches(xx,yy,ids):
        check_time(); a,b=ready(clean_batch(model,state,x,y,mask)); finite((a,b))
        acc+=float(a); ce+=float(b)
    return dict(accuracy=100*acc/len(ids),ce=ce/len(ids),n=len(ids))


def validate_igr_math():
    # Independent analytic mixed-derivative check, catches a detached penalty or
    # accidental gradient-of-batch-mean / batch-size scaling.
    x=jnp.array([[.2,.7],[.5,.1]],dtype=jnp.float32); w=jnp.array([.3,-.4],dtype=jnp.float32)
    def toy(ww,xx): return .5*jnp.square(xx@ww)
    def p(ww,xx):
        dx=jax.grad(lambda z:jnp.sum(toy(ww,z)))(xx)
        return jnp.mean(jnp.sum(dx*dx,axis=1))
    expected=jnp.mean(jnp.square(x@w))*jnp.sum(w*w)
    mixed=2*jnp.mean((x@w)[:,None]*x,axis=0)*jnp.sum(w*w)+2*w*jnp.mean(jnp.square(x@w))
    np.testing.assert_allclose(np.asarray(p(w,x)),np.asarray(expected),rtol=1e-5)
    np.testing.assert_allclose(np.asarray(jax.grad(p)(w,x)),np.asarray(mixed),rtol=1e-5,atol=1e-7)
    np.testing.assert_allclose(np.asarray(p(w,x)),np.asarray(p(w,jnp.concatenate([x,x]))),rtol=1e-5)
    assert float(ready(l2(jax.grad(p)(w,x))))>0
    return dict(toy_penalty=float(p(w,x)),mixed_gradient=np.asarray(mixed).tolist(),batch_duplication_invariant=True)

METHOD=PHASE
assert METHOD in METHODS
assert C['epochs']==1 and C['train_per_class']==450 and B==128
assert C['schedule_epochs']==100

def state_hash(model,state):
    digest=hashlib.sha256()
    for v in jax.tree_leaves((model.params,state)):
        a=np.asarray(v)
        digest.update(str((a.shape,str(a.dtype))).encode()); digest.update(a.tobytes())
    return digest.hexdigest()

def step_statistics(values):
    a=np.asarray(values,dtype=np.float64)
    return dict(mean=float(np.mean(a)),median=float(np.median(a)),p10=float(np.percentile(a,10)),
                p90=float(np.percentile(a,90)),maximum=float(np.max(a)),first=float(a[0]))

def run_one_epoch(method):
    # Warm-up and short timing use a sacrificial model. All its state is discarded.
    t=time.perf_counter();opt,state=fresh(C['seed']);ready((opt,state))
    initial_hash=state_hash(opt.target,state); init_s=time.perf_counter()-t
    checks=validate_igr_math() if method=='IGR' else {}
    initial_model=opt.target
    warm=[]; probe=[]; it=batches(0,C['seed'])
    t=time.perf_counter()
    for _ in range(C['warmup_steps']):
        check_time(); tick=time.perf_counter();x,y=next(it)
        opt,state,metrics=ready(STEPS_FN[method](opt,state,x,y,np.float32(C['lr']),np.float32(C['igr_lambda'])))
        finite(metrics);warm.append(time.perf_counter()-tick)
    warm_s=time.perf_counter()-t
    change=float(ready(l2(sub(opt.target,initial_model))))
    assert change>0 and np.isfinite(change)
    if method=='IGR':assert float(metrics['igr_penalty'])>0
    for _ in range(C['timed_steps']):
        check_time();tick=time.perf_counter();x,y=next(it)
        opt,state,metrics=ready(STEPS_FN[method](opt,state,x,y,np.float32(C['lr']),np.float32(C['igr_lambda'])))
        finite(metrics);probe.append(time.perf_counter()-tick)
    stats=step_statistics(probe)
    stable=(stats['maximum']<=3*max(stats['median'],1e-12))
    warm_memory=gpu_memory()
    save(method+'_warmup.json',dict(warmup_seconds=warm_s,warmup_step_seconds=warm,
        timed_step_seconds=probe,timed_step_statistics_seconds=stats,timed_probe_stable=stable,
        parameter_change_l2=change,igr_checks=checks,gpu_memory_snapshot=warm_memory))
    del opt,state,initial_model;gc.collect()

    # Fresh seeded initialization and fresh epoch generator. Warm-up contributes NO updates.
    tick=time.perf_counter();opt,state=fresh(C['seed']);ready((opt,state))
    assert state_hash(opt.target,state)==initial_hash, 'Reset did not reproduce initial model/BN'
    reset_s=time.perf_counter()-tick
    order=np.random.RandomState(C['seed']+1000).permutation(ti)
    order_hash=hashlib.sha256(order.tobytes()).hexdigest()
    it=batches(0,C['seed']); step_times=[];prep_times=[];compute_times=[];trace=[]
    ce_sum=penalty_sum=0.;epoch_start=time.perf_counter()
    for i in range(STEPS):
        check_time(); tick=time.perf_counter();x,y=next(it);prepared=time.perf_counter()
        # Epoch 1 of the proposed 100-epoch cosine schedule, not a one-epoch decay.
        lr=np.float32(C['lr']*.5*(1+math.cos(math.pi*i/(C['schedule_epochs']*STEPS))))
        opt,state,metrics=ready(STEPS_FN[method](opt,state,x,y,lr,np.float32(C['igr_lambda'])))
        end=time.perf_counter();finite(metrics)
        prep_times.append(prepared-tick);compute_times.append(end-prepared);step_times.append(end-tick)
        ce_sum+=float(metrics['clean_ce']);penalty_sum+=float(metrics['igr_penalty'])
        if i<5:trace.append(dict(step=i+1,lr=float(lr),seconds=step_times[-1]))
        if (i+1)%50==0:
            print(method,'epoch 1:',i+1,'/',STEPS,'updates',flush=True)
            save(method+'_progress.json',dict(completed_steps=i+1,total_steps=STEPS,epoch=1))
    epoch_s=time.perf_counter()-epoch_start
    assert len(step_times)==STEPS==351
    try:next(it)
    except StopIteration:pass
    else:raise AssertionError('Expected exactly one epoch')
    train_memory=gpu_memory()
    save(method+'_steps.json',dict(step_seconds=step_times,input_preparation_seconds=prep_times,
        synchronized_update_seconds=compute_times,first_steps=trace,epoch_training_seconds=epoch_s))

    # Compile validation outside its measured window. Only validation, never test, is scored.
    x,y,mask=next(eval_batches(images,labels,vi));tick=time.perf_counter()
    ready(clean_batch(opt.target,state,x,y,mask));eval_compile_s=time.perf_counter()-tick
    tick=time.perf_counter();validation=evaluate(opt.target,state,images,labels,vi);val_s=time.perf_counter()-tick
    checkpoint_s=restore_s=0.;checkpoint_bytes=0
    if C['measure_checkpoint']:
        payload=dict(optimizer=opt,state=state,completed_epoch=1)
        dest=OUT/(method+'_epoch1.msgpack');tmp=dest.with_name(dest.name+'.tmp')
        tick=time.perf_counter();tmp.write_bytes(serialization.to_bytes(payload));tmp.replace(dest)
        checkpoint_s=time.perf_counter()-tick;checkpoint_bytes=dest.stat().st_size
        tick=time.perf_counter();restored=serialization.from_bytes(payload,dest.read_bytes())
        for old,new in zip(jax.tree_leaves(payload),jax.tree_leaves(restored)):
            np.testing.assert_array_equal(np.asarray(old),np.asarray(new))
        restore_s=time.perf_counter()-tick;del restored,payload
    wall=time.monotonic()-BEGIN
    # Includes imports/data load/init/warm-up/probe/reset/eval compilation and optional restore,
    # once per fresh process. Projection deliberately retains this cost conservatively.
    overhead=max(0.,wall-epoch_s-val_s-checkpoint_s)
    result=dict(method=method,epochs_measured=1,schedule_epochs=C['schedule_epochs'],seed=C['seed'],
        igr_lambda=C['igr_lambda'] if method=='IGR' else 0.,batch_size=B,train_examples=len(ti),
        validation_examples=len(vi),steps=STEPS,examples_used=STEPS*B,dropped_examples=len(ti)-STEPS*B,
        parameter_count=sum(v.size for v in jax.tree_leaves(opt.target.params)),
        initial_model_bn_sha256=initial_hash,epoch_order_sha256=order_hash,
        epoch_training_seconds=epoch_s,step_statistics_seconds=step_statistics(step_times),
        summed_input_preparation_seconds=sum(prep_times),summed_synchronized_update_seconds=sum(compute_times),
        initialization_seconds=init_s,warmup_seconds=warm_s,reset_seconds=reset_s,
        warmup_probe_stable=stable,validation_compile_seconds=eval_compile_s,
        validation_seconds=val_s,validation_accuracy_diagnostic_only=validation['accuracy'],
        validation_ce_diagnostic_only=validation['ce'],train_clean_ce=ce_sum/STEPS,
        train_igr_penalty=penalty_sum/STEPS,checkpoint_seconds=checkpoint_s,
        checkpoint_bytes=checkpoint_bytes,checkpoint_restore_check_seconds=restore_s,
        fixed_worker_overhead_seconds=overhead,worker_wall_seconds=wall,
        gpu_memory_after_warmup=warm_memory,gpu_memory_after_epoch=train_memory,
        memory_measurement='nvidia-smi snapshots only; not measured peak memory',
        worker_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
        source_revision=C['source_revision'],gpu=C['gpu'],
        environment=dict(python=platform.python_version(),jax=jax.__version__,flax=flax.__version__,
                         numpy=np.__version__,tensorflow=tf.__version__),status='complete')
    save(method+'_result.json',result)
    print('MEASURED ONE EPOCH',json.dumps(result,indent=2),flush=True)
    return result

try:
    save(METHOD+'_status.json',dict(status='running'))
    assert jax.__version__=='0.2.6' and flax.__version__=='0.2.2'
    live_gpu=subprocess.check_output(['nvidia-smi','-i','0','--query-gpu=name,memory.total',
        '--format=csv,noheader,nounits'],text=True,timeout=15).strip()
    assert live_gpu==C['gpu'] and 'A100' in live_gpu, 'GPU changed since setup'
    run_one_epoch(METHOD)
    save(METHOD+'_status.json',dict(status='complete',wall_seconds=time.monotonic()-BEGIN))
except BaseException as exc:
    save(METHOD+'_status.json',dict(status='failed',error=str(exc),traceback=traceback.format_exc(),
        wall_seconds=time.monotonic()-BEGIN,gpu_memory_snapshot=gpu_memory()))
    raise

'''
compile(WORKER_SOURCE,'<A100 worker>','exec')


In [ ]:
CFG=dict(epochs=1,schedule_epochs=100,train_per_class=450,batch_size=128,seed=42,split_seed=2026,
    lr=0.1,momentum=0.9,weight_decay=0.0005,clip=5.0,rho=0.05,igr_lambda=0.01,
    warmup_steps=10,timed_steps=20,measure_checkpoint=MEASURE_CHECKPOINT,
    worker_seconds=MAX_SESSION_MINUTES*60,repo=str(REPO),output=str(SESSION),
    data_dir=str(ROOT/'tensorflow_datasets'),source_revision=SOURCE_REVISION,gpu=GPU,
    protocol='cifar100_resnet18_a100_one_epoch_timing_v1')
assert CFG['epochs']==1 and CFG['batch_size']==128 and CFG['train_per_class']==450
compile(WORKER_SOURCE,'<A100 timing worker>','exec')
SCRIPT=SESSION/'timing_worker.py';SCRIPT.write_text(WORKER_SOURCE)
atomic_json(SESSION/'config.json',CFG)
MANIFEST=dict(worker_sha256=hashlib.sha256(SCRIPT.read_bytes()).hexdigest(),
    config_sha256=hashlib.sha256((SESSION/'config.json').read_bytes()).hexdigest())
atomic_json(SESSION/'manifest.json',MANIFEST)
DATA_SOURCE=r"""
import os,time,json,sys
from pathlib import Path
import tensorflow as tf
tf.config.experimental.set_visible_devices([], 'GPU')
import tensorflow_datasets as tfds
start=time.monotonic();c=json.loads(Path(sys.argv[1]).read_text())
builder=tfds.builder('cifar100:3.0.2',data_dir=c['data_dir']);builder.download_and_prepare()
assert builder.info.splits['train'].num_examples==50000 and builder.info.splits['test'].num_examples==10000
Path(c['output'],'data_setup.json').write_text(json.dumps(dict(seconds=time.monotonic()-start,
    train_available=50000,validation_held_out=5000,test_loaded=False),indent=2))
print('Dataset cache ready. Official test images were not loaded.',flush=True)
"""
compile(DATA_SOURCE,'<dataset setup>','exec')
data_script=SESSION/'prepare_data.py';data_script.write_text(DATA_SOURCE)
t=time.monotonic()
try:command([PYTHON,data_script,SESSION/'config.json'],SESSION/'data_setup.log',ENV,DEADLINE)
finally:atomic_json(SESSION/'data_host_timing.json',dict(seconds=time.monotonic()-t))


## 3. Measure one complete epoch per method
Runs SGD, then SAM, then IGR. Records full synchronized step timings (including host augmentation and transfers), first steps, input-preparation and update time, compilation/warm-up, complete epoch wall time, validation and optional checkpoint cost. The projection uses the **measured full epoch**, never the old five-step mean. Short-probe stability is reported, but is not substituted for the actual epoch.

These are one-seed, first-epoch hardware measurements. GPU-memory snapshots are not peak measurements. Validation accuracy is a diagnostic of a barely trained model, not a generalization finding. A failed/partial method cannot enter the full-study projection.


In [ ]:
def launch_timing(method):
    assert method in ('SGD','SAM','IGR')
    assert hashlib.sha256(SCRIPT.read_bytes()).hexdigest()==MANIFEST['worker_sha256']
    assert hashlib.sha256((SESSION/'config.json').read_bytes()).hexdigest()==MANIFEST['config_sha256']
    marker=SESSION/(method+'_launch.json')
    assert not marker.exists(), 'This attempt is preserved. Rerun setup for a fresh timing folder.'
    atomic_json(marker,dict(status='running',manifest=MANIFEST))
    start=time.monotonic();status='failed'
    try:
        command([PYTHON,'-u',SCRIPT,SESSION/'config.json',method],SESSION/(method+'.log'),ENV,DEADLINE)
        assert json.loads((SESSION/(method+'_status.json')).read_text())['status']=='complete'
        result=json.loads((SESSION/(method+'_result.json')).read_text())
        assert result['epochs_measured']==1 and result['steps']==351 and result['status']=='complete'
        status='complete'
    finally:atomic_json(marker,dict(status=status,manifest=MANIFEST,wall_seconds=time.monotonic()-start))

if RUN_TIMING:
    # Sequential fresh processes release each method's GPU state before the next starts.
    for method in ('SGD','SAM','IGR'):launch_timing(method)
    print('Finished exactly one epoch per method; no 100-epoch training was started.')
else:print('Timing disabled. Set RUN_TIMING=True to run one epoch per method.')


## 4. Measured times and full-study planning
The planning scenario defaults to 100 epochs × 3 seeds × 3 final methods, plus three additional 100-epoch IGR tuning trials on one development seed. If you intend a different search horizon, change the **planning values only** before generating the report. The IGR search is conservatively additional to final training; no unmeasured reuse credit is assumed.

Every projected epoch includes a full validation pass and a checkpoint write when enabled. Measured process/warm-up/probe/reset overhead is added once per projected run, conservatively. Setup/data preparation are recorded separately. The additional-evaluation reserve is an explicit unmeasured allowance, not a timing for PGD/sharpness. A further 20% allowance is applied to the projected sum. First-epoch extrapolation does not establish sustained speed or total cloud cost. No SAM+IGR optional arm or extra models are included.

The report produces CSV and JSON. It refuses a combined projection if methods are missing, initial states/orders differ, or hardware/source differs. It only reports whether the scenario fits the declared 35-hour planning budget; it cannot read your live allocation balance.


In [ ]:
REPORT_SOURCE = r'''

def project_budget(rows,plan):
    assert set(r['method'] for r in rows)=={'SGD','SAM','IGR'} and len(rows)==3
    assert all(r['status']=='complete' and r['epochs_measured']==1 and r['steps']==351 for r in rows)
    assert len(set(r['initial_model_bn_sha256'] for r in rows))==1, 'Initial states differ'
    assert len(set(r['epoch_order_sha256'] for r in rows))==1, 'Training orders differ'
    assert len(set(r['worker_sha256'] for r in rows))==1, 'Workers differ'
    assert len(set((r['gpu'],r['batch_size']) for r in rows))==1, 'Hardware/batch differs'
    assert plan['epochs']>0 and plan['seeds']>=3 and 0<=plan['allowance']<=1
    assert plan['tuning_trials']>=0 and plan['tuning_epochs']>=0 and plan['tuning_seeds']>=0
    costs=[]
    for r in rows:
        per_epoch=r['epoch_training_seconds']+r['validation_seconds']+r['checkpoint_seconds']
        one_run=r['fixed_worker_overhead_seconds']+plan['epochs']*per_epoch
        costs.append(dict(method=r['method'],projected_per_seed_hours=one_run/3600,
            projected_all_seeds_hours=plan['seeds']*one_run/3600))
    igr=next(r for r in rows if r['method']=='IGR')
    tuning=plan['tuning_trials']*plan['tuning_seeds']*(igr['fixed_worker_overhead_seconds']+
        plan['tuning_epochs']*(igr['epoch_training_seconds']+igr['validation_seconds']+igr['checkpoint_seconds']))
    main=sum(r['projected_all_seeds_hours'] for r in costs)
    base=main+tuning/3600+plan['extra_evaluation_minutes']/60
    return dict(estimated_not_measured=True,plan=plan,per_method=costs,
        projected_main_hours=main,projected_additional_igr_search_hours=tuning/3600,
        unmeasured_extra_evaluation_reserve_hours=plan['extra_evaluation_minutes']/60,
        projected_total_before_allowance_hours=base,projected_total_with_allowance_hours=base*(1+plan['allowance']),
        fits_declared_planning_budget=base*(1+plan['allowance'])<=plan['budget_hours'],
        scope='Epoch-1 extrapolation; same data/batch/FP32/update/validation/checkpoint protocol only. No automatic full training.')

import json,sys,csv
from pathlib import Path
p=Path(sys.argv[1]);plan=json.loads(Path(sys.argv[2]).read_text())
rows=[]
for method in ('SGD','SAM','IGR'):
    f=p/(method+'_result.json')
    if f.exists():rows.append(json.loads(f.read_text()))
if not rows:
    print('No completed epochs. Inspect setup/data/method logs; there is no measured budget yet.');sys.exit(0)
fields=['method','epoch_training_seconds','validation_seconds','checkpoint_seconds','fixed_worker_overhead_seconds','worker_wall_seconds']
with (p/'measured_epoch_times.csv').open('w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=fields);w.writeheader();w.writerows({k:r[k] for k in fields} for r in rows)
for r in rows:
    print('%s: train %.2f s | validation %.2f s | checkpoint %.2f s | other worker overhead %.2f s | worker total %.2f s' %
        tuple(r[k] for k in fields))
    print('  step median %.4f s, p90 %.4f s; warm-up probe stable: %s' %
        (r['step_statistics_seconds']['median'],r['step_statistics_seconds']['p90'],r['warmup_probe_stable']))
if len(rows)!=3:
    print('INCOMPLETE: only %d/3 methods completed. No full-study projection generated.'%len(rows));sys.exit(0)
result=project_budget(rows,plan)
(p/'budget_projection.json').write_text(json.dumps(result,indent=2,allow_nan=False))
print('\nPROJECTED full experiment (not measured):')
for r in result['per_method']:print(r['method'],round(r['projected_all_seeds_hours'],2),'hours across',plan['seeds'],'seeds')
print('Main comparison:',round(result['projected_main_hours'],2),'hours')
print('Additional IGR search:',round(result['projected_additional_igr_search_hours'],2),'hours')
print('Unmeasured additional evaluation reserve:',result['unmeasured_extra_evaluation_reserve_hours'],'hours')
print('Total with allowance:',round(result['projected_total_with_allowance_hours'],2),'hours')
print('Within declared planning budget:',result['fits_declared_planning_budget'])
print('Projection only. No full experiment was launched. One-epoch validation accuracy is not a scientific result.')

'''

compile(REPORT_SOURCE,'<A100 timing report>','exec')
PLAN=dict(epochs=PLANNED_EPOCHS,seeds=PLANNED_SEEDS,tuning_trials=PLANNED_IGR_SEARCH_TRIALS,
    tuning_epochs=PLANNED_IGR_SEARCH_EPOCHS,tuning_seeds=PLANNED_IGR_SEARCH_SEEDS,
    extra_evaluation_minutes=EXTRA_EVALUATION_RESERVE_MINUTES,allowance=ALLOWANCE_FRACTION,
    budget_hours=PLANNING_BUDGET_HOURS)
atomic_json(SESSION/'planning_scenario.json',PLAN)
report_file=SESSION/'timing_report.py';report_file.write_text(REPORT_SOURCE)
# CPU-only reporting remains available after a GPU timeout; it starts no training.
command([PYTHON,report_file,SESSION,SESSION/'planning_scenario.json'],SESSION/'report.log',ENV,
    deadline=time.monotonic()+180)


## 5. Download timing evidence
The ZIP includes the exact worker/configuration/source hashes, actual per-step/epoch measurements, logs and failure statuses, environment information, split indices and budget scenario/projection. Large optimizer checkpoints are excluded. Download before closing the session, or use Drive persistence from the beginning.

If a cell fails, you can still run reporting/export to preserve partial evidence. There is no automatic training resume in this benchmark: a new attempt starts a fresh one-epoch measurement, while dataset/environment caches are reused. Existing notebooks and results are untouched.


In [ ]:
import zipfile
archive=ROOT/(SESSION.name+'_a100_one_epoch_timing.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in SESSION.rglob('*'):
        if f.is_file() and f.suffix not in ('.msgpack','.tmp'):
            z.write(f,str(Path(SESSION.name)/f.relative_to(SESSION)))
print('Download:',archive)
try:
    from google.colab import files
except ImportError:
    print('Use the Jupyter file browser to download the ZIP from the printed path.')
else:files.download(str(archive))
